# Country-year electricity-system and policy controls

This notebook prepares country-year controls for Cox robustness checks responding to reviewer concerns about endogenous data-center siting. The final output is keyed by `Alpha_3_code` and `year`.

In [ ]:
import os
import re
import warnings
from functools import reduce
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# Path settings
BASE_PATH = Path.cwd().parent.parent

RAW = BASE_PATH / 'Data' / 'raw'
TEMP = BASE_PATH / 'Data' / 'temp'
USE = BASE_PATH / 'Data' / 'use'
FIGURES = BASE_PATH / 'Results' / 'Figures'
TABLES = BASE_PATH / 'Results' / 'Tables'

for path in [RAW, TEMP, USE, FIGURES, TABLES]:
    path.mkdir(parents=True, exist_ok=True)

PRICE_INPUT = RAW / 'price_of_electricty_Data_Extract_From_Doing_Business.xlsx'
TD_LOSS_INPUT = RAW / 'API_EG.ELC.LOSS.ZS_DS2_en_csv_v2_3166.csv'
EMBER_INPUT = RAW / 'yearly_full_release_long_format.csv'
NETZERO_INPUT = RAW / 'NET-ZERO-current_snapshot_2026-05-26_06-40-23.xlsx'
POLICY_INPUT = RAW / 'climate_policy_database_policies_export.csv'

OUTPUT_CSV = TEMP / 'country_year_electricity_policy_controls.csv'
OUTPUT_DTA = TEMP / 'country_year_electricity_policy_controls.dta'
OUTPUT_USE_DTA = USE / 'country_year_electricity_policy_controls.dta'
DICT_OUTPUT = TEMP / 'country_year_electricity_policy_controls_dictionary.csv'
COVERAGE_OUTPUT = TEMP / 'country_year_electricity_policy_controls_coverage.csv'

YEAR_MIN = 2000
YEAR_MAX = 2024

print(f'BASE_PATH: {BASE_PATH}')
print(f'Output CSV: {OUTPUT_CSV}')
print(f'Output DTA: {OUTPUT_DTA}')

In [ ]:
metadata_rows = []

def add_meta(variable, source, description, construction='', unit='', notes=''):
    metadata_rows.append({
        'variable': variable,
        'source': source,
        'description': description,
        'construction': construction,
        'unit': unit,
        'notes': notes,
    })


def clean_iso3(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().upper()
    if x in ['', '..', 'NAN', 'NONE']:
        return np.nan
    match = re.match(r'^([A-Z]{3})', x)
    return match.group(1) if match else np.nan


def to_num(s):
    return pd.to_numeric(s.replace('..', np.nan) if hasattr(s, 'replace') else s, errors='coerce')


def safe_stata_name(name):
    if len(name) > 32:
        raise ValueError(f'Variable name exceeds Stata limit: {name} ({len(name)})')
    if not re.match(r'^[A-Za-z_][A-Za-z0-9_]*$', name):
        raise ValueError(f'Variable name is not Stata-safe: {name}')
    return name


def infer_country_year_bounds(*frames):
    pieces = []
    for frame in frames:
        if frame is None or frame.empty:
            continue
        if {'Alpha_3_code', 'year'}.issubset(frame.columns):
            tmp = frame[['Alpha_3_code', 'year']].copy()
            tmp['Alpha_3_code'] = tmp['Alpha_3_code'].astype(str)
            tmp['year'] = pd.to_numeric(tmp['year'], errors='coerce')
            tmp = tmp.dropna(subset=['Alpha_3_code', 'year'])
            tmp = tmp[tmp['Alpha_3_code'].str.fullmatch(r'[A-Z]{3}', na=False)]
            pieces.append(tmp)
    if not pieces:
        return pd.DataFrame(columns=['Alpha_3_code', 'min_year', 'max_year'])
    years = pd.concat(pieces, ignore_index=True)
    bounds = years.groupby('Alpha_3_code', as_index=False).agg(
        min_year=('year', 'min'),
        max_year=('year', 'max')
    )
    bounds['min_year'] = bounds['min_year'].astype(int)
    bounds['max_year'] = bounds['max_year'].astype(int)
    return bounds


def make_country_specific_panel(bounds):
    rows = []
    for row in bounds.itertuples(index=False):
        for yr in range(int(row.min_year), int(row.max_year) + 1):
            rows.append((row.Alpha_3_code, yr))
    return pd.DataFrame(rows, columns=['Alpha_3_code', 'year'])


def make_panel(alpha_codes, year_min=YEAR_MIN, year_max=YEAR_MAX):
    # Generic balanced panel helper retained for policy cumulative calculations.
    alpha_codes = sorted([c for c in set(alpha_codes) if isinstance(c, str) and re.match(r'^[A-Z]{3}$', c)])
    years = list(range(year_min, year_max + 1))
    idx = pd.MultiIndex.from_product([alpha_codes, years], names=['Alpha_3_code', 'year'])
    return idx.to_frame(index=False)

def add_lags(df, cols, group_col='Alpha_3_code', time_col='year', lag=1):
    df = df.sort_values([group_col, time_col]).copy()
    for col in cols:
        if col in df.columns:
            lag_col = safe_stata_name(f'l{lag}_{col}')
            df[lag_col] = df.groupby(group_col, sort=False)[col].shift(lag)
            add_meta(
                lag_col,
                'Derived',
                f'One-year lag of {col}.',
                f'Within-country lag of {col} by calendar year.',
                notes='Use lagged controls when contemporaneous electricity-system conditions may be jointly determined.'
            )
    return df


def build_coverage(df):
    rows = []
    for col in df.columns:
        if col in ['Alpha_3_code', 'year']:
            continue
        s = df[col]
        nonmissing = s.notna()
        rows.append({
            'variable': col,
            'nonmissing_rows': int(nonmissing.sum()),
            'nonmissing_share': float(nonmissing.mean()),
            'countries_nonmissing': int(df.loc[nonmissing, 'Alpha_3_code'].nunique()),
            'min_year_nonmissing': int(df.loc[nonmissing, 'year'].min()) if nonmissing.any() else np.nan,
            'max_year_nonmissing': int(df.loc[nonmissing, 'year'].max()) if nonmissing.any() else np.nan,
        })
    return pd.DataFrame(rows)

print('Helper functions ready.')


## 1. World Bank Doing Business electricity price

The raw file reports prices in US cents/kWh. This notebook keeps only national rows whose `Country Code` is exactly a three-letter ISO code. City-coded rows such as `CHN_BEI` are excluded rather than truncated, so the control remains a consistent country-year measure.


In [ ]:
def load_electricity_price():
    price = pd.read_excel(PRICE_INPUT, sheet_name='Data')
    year_cols = [c for c in price.columns if re.search(r'\[YR\d{4}\]', str(c))]

    keep = ['Country Name', 'Country Code'] + year_cols
    price = price[keep].copy()
    price['Country Code'] = price['Country Code'].astype(str).str.strip().str.upper()

    city_like_rows = price['Country Code'].str.contains('_', na=False).sum()
    price = price[price['Country Code'].str.fullmatch(r'[A-Z]{3}', na=False)].copy()
    price['Alpha_3_code'] = price['Country Code']

    long = price.melt(
        id_vars=['Country Name', 'Country Code', 'Alpha_3_code'],
        value_vars=year_cols,
        var_name='year_label',
        value_name='elec_price_db_uscent_kwh_raw'
    )
    long['year'] = long['year_label'].str.extract(r'(\d{4})').astype(int)
    long['elec_price_db_uscent_kwh_raw'] = pd.to_numeric(
        long['elec_price_db_uscent_kwh_raw'].replace('..', np.nan),
        errors='coerce'
    )

    grouped = long.groupby(['Alpha_3_code', 'year'], as_index=False).agg(
        elec_price_db_uscent_kwh_raw=('elec_price_db_uscent_kwh_raw', 'mean'),
        elec_price_db_n_obs=('elec_price_db_uscent_kwh_raw', 'count'),
        country_name_price=('Country Name', lambda x: '; '.join(sorted(set(map(str, x)))[:5]))
    )

    add_meta('elec_price_db_uscent_kwh_raw', 'World Bank Doing Business',
             'Price of electricity for a standardized warehouse case.',
             'Only rows with a pure three-letter ISO country code are retained; city-coded rows with suffixes are excluded.',
             'US cents per kWh',
             'Missing values are left as missing; no interpolation, forward-fill, or back-fill is applied in this notebook.')
    add_meta('elec_price_db_n_obs', 'World Bank Doing Business',
             'Number of national observations used for the country-year electricity price value.',
             'Count of non-missing raw price observations after keeping only pure ISO-3 country rows.')

    print(
        f'Electricity price controls: {grouped.shape[0]:,} country-year rows, '
        f'{grouped.Alpha_3_code.nunique():,} countries; excluded city-coded rows: {city_like_rows:,}'
    )
    return grouped

price_controls = load_electricity_price()
price_controls.head()


## 1b. World Bank transmission and distribution losses

The World Bank WDI file is read from the fifth row header. The notebook keeps country/economy rows and excludes World Bank aggregate regions using `Country Name`, so regional aggregates such as `World`, `High income`, or `Europe & Central Asia` are not treated as countries.


In [ ]:
WB_AGGREGATE_COUNTRY_NAMES = {
    'Africa Eastern and Southern',
    'Africa Western and Central',
    'Arab World',
    'Caribbean small states',
    'Central Europe and the Baltics',
    'Early-demographic dividend',
    'East Asia & Pacific',
    'East Asia & Pacific (excluding high income)',
    'East Asia & Pacific (IDA & IBRD countries)',
    'Euro area',
    'Europe & Central Asia',
    'Europe & Central Asia (excluding high income)',
    'Europe & Central Asia (IDA & IBRD countries)',
    'European Union',
    'Fragile and conflict affected situations',
    'Heavily indebted poor countries (HIPC)',
    'High income',
    'IBRD only',
    'IDA & IBRD total',
    'IDA blend',
    'IDA only',
    'IDA total',
    'Late-demographic dividend',
    'Latin America & Caribbean',
    'Latin America & Caribbean (excluding high income)',
    'Latin America & the Caribbean (IDA & IBRD countries)',
    'Least developed countries: UN classification',
    'Low & middle income',
    'Low income',
    'Lower middle income',
    'Middle East & North Africa',
    'Middle East & North Africa (excluding high income)',
    'Middle East & North Africa (IDA & IBRD countries)',
    'Middle income',
    'North America',
    'Not classified',
    'OECD members',
    'Other small states',
    'Pacific island small states',
    'Post-demographic dividend',
    'Pre-demographic dividend',
    'Small states',
    'South Asia',
    'South Asia (IDA & IBRD)',
    'Sub-Saharan Africa',
    'Sub-Saharan Africa (excluding high income)',
    'Sub-Saharan Africa (IDA & IBRD countries)',
    'Upper middle income',
    'World',
}


def load_td_losses_controls():
    losses = pd.read_csv(TD_LOSS_INPUT, skiprows=4)
    losses = losses.loc[:, ~losses.columns.astype(str).str.startswith('Unnamed')].copy()
    losses['Country Name'] = losses['Country Name'].astype(str).str.strip()
    losses['Country Code'] = losses['Country Code'].astype(str).str.strip().str.upper()

    indicator_codes = sorted(losses['Indicator Code'].dropna().astype(str).unique())
    if indicator_codes != ['EG.ELC.LOSS.ZS']:
        print(f'T&D losses file contains indicator codes: {indicator_codes}')

    aggregate_rows = losses['Country Name'].isin(WB_AGGREGATE_COUNTRY_NAMES).sum()
    losses = losses[
        losses['Country Code'].str.fullmatch(r'[A-Z]{3}', na=False)
        & ~losses['Country Name'].isin(WB_AGGREGATE_COUNTRY_NAMES)
    ].copy()
    losses['Alpha_3_code'] = losses['Country Code']

    year_cols = [c for c in losses.columns if re.fullmatch(r'\d{4}', str(c))]
    long = losses.melt(
        id_vars=['Country Name', 'Country Code', 'Alpha_3_code', 'Indicator Name', 'Indicator Code'],
        value_vars=year_cols,
        var_name='year',
        value_name='td_loss_pct_output'
    )
    long['year'] = long['year'].astype(int)
    long['td_loss_pct_output'] = pd.to_numeric(long['td_loss_pct_output'].replace('', np.nan), errors='coerce')
    grouped = long.groupby(['Alpha_3_code', 'year'], as_index=False).agg(
        td_loss_pct_output=('td_loss_pct_output', 'mean'),
        country_name_td_loss=('Country Name', lambda x: '; '.join(sorted(set(map(str, x)))[:5]))
    )

    add_meta('td_loss_pct_output', 'World Bank WDI',
             'Electric power transmission and distribution losses as percent of output.',
             'Indicator EG.ELC.LOSS.ZS; country/economy rows only, World Bank aggregate regions excluded by Country Name.',
             '% of electricity output')
    add_meta('country_name_td_loss', 'World Bank WDI',
             'Country name in WDI transmission and distribution losses file.',
             'Retained for auditability after excluding aggregate Country Name rows.')

    print(
        f'T&D loss controls: {grouped.shape[0]:,} country-year rows, '
        f'{grouped.Alpha_3_code.nunique():,} countries; excluded aggregate rows: {aggregate_rows:,}'
    )
    return grouped

td_loss_controls = load_td_losses_controls()
td_loss_controls.head()


## 2. Ember electricity-system controls

These controls address load growth, renewable penetration, fossil/coal/gas reliance, generation mix, capacity mix, imports, and power-sector emissions intensity.

In [ ]:
EMBER_SPECS = [
    ('demand_twh', 'Electricity demand', 'Demand', 'Demand', 'TWh', 'Value'),
    ('demand_yoy_abs_twh', 'Electricity demand', 'Demand', 'Demand', 'TWh', 'YoY absolute change'),
    ('demand_yoy_pct', 'Electricity demand', 'Demand', 'Demand', 'TWh', 'YoY % change'),
    ('demand_pc_mwh', 'Electricity demand', 'Demand per capita', 'Demand per capita', 'MWh', 'Value'),
    ('generation_twh', 'Electricity generation', 'Total', 'Total Generation', 'TWh', 'Value'),
    ('net_imports_twh', 'Electricity imports', 'Electricity imports', 'Net Imports', 'TWh', 'Value'),
    ('renew_gen_share_pct', 'Electricity generation', 'Aggregate fuel', 'Renewables', '%', 'Value'),
    ('wind_solar_gen_share_pct', 'Electricity generation', 'Aggregate fuel', 'Wind and Solar', '%', 'Value'),
    ('clean_gen_share_pct', 'Electricity generation', 'Aggregate fuel', 'Clean', '%', 'Value'),
    ('fossil_gen_share_pct', 'Electricity generation', 'Aggregate fuel', 'Fossil', '%', 'Value'),
    ('coal_gen_share_pct', 'Electricity generation', 'Fuel', 'Coal', '%', 'Value'),
    ('gas_gen_share_pct', 'Electricity generation', 'Fuel', 'Gas', '%', 'Value'),
    ('renew_gen_twh', 'Electricity generation', 'Aggregate fuel', 'Renewables', 'TWh', 'Value'),
    ('wind_solar_gen_twh', 'Electricity generation', 'Aggregate fuel', 'Wind and Solar', 'TWh', 'Value'),
    ('clean_gen_twh', 'Electricity generation', 'Aggregate fuel', 'Clean', 'TWh', 'Value'),
    ('fossil_gen_twh', 'Electricity generation', 'Aggregate fuel', 'Fossil', 'TWh', 'Value'),
    ('coal_gen_twh', 'Electricity generation', 'Fuel', 'Coal', 'TWh', 'Value'),
    ('gas_gen_twh', 'Electricity generation', 'Fuel', 'Gas', 'TWh', 'Value'),
    ('total_emissions_mtco2', 'Power sector emissions', 'Total', 'Total emissions', 'mtCO2', 'Value'),
    ('co2_intensity_g_kwh', 'Power sector emissions', 'CO2 intensity', 'CO2 intensity', 'gCO2/kWh', 'Value'),
    ('coal_capacity_gw', 'Capacity', 'Fuel', 'Coal', 'GW', 'Value'),
    ('gas_capacity_gw', 'Capacity', 'Fuel', 'Gas', 'GW', 'Value'),
    ('renew_capacity_gw', 'Capacity', 'Aggregate fuel', 'Renewables', 'GW', 'Value'),
    ('clean_capacity_gw', 'Capacity', 'Aggregate fuel', 'Clean', 'GW', 'Value'),
    ('fossil_capacity_gw', 'Capacity', 'Aggregate fuel', 'Fossil', 'GW', 'Value'),
]

for var, category, subcategory, variable, unit, value_col in EMBER_SPECS:
    safe_stata_name(var)


def load_ember_controls():
    usecols = ['Area', 'ISO 3 code', 'Year', 'Area type', 'Category', 'Subcategory', 'Variable', 'Unit', 'Value', 'YoY absolute change', 'YoY % change']
    ember = pd.read_csv(EMBER_INPUT, usecols=usecols)
    ember = ember[ember['Area type'].eq('Country or economy')].copy()
    ember['Alpha_3_code'] = ember['ISO 3 code'].map(clean_iso3)
    ember = ember.dropna(subset=['Alpha_3_code']).copy()
    ember = ember[(ember['Year'] >= YEAR_MIN) & (ember['Year'] <= YEAR_MAX)].copy()

    frames = []
    country_names = ember[['Alpha_3_code', 'Area']].drop_duplicates('Alpha_3_code').rename(columns={'Area': 'country_name_ember'})

    for outvar, category, subcategory, variable, unit, value_col in EMBER_SPECS:
        m = (
            ember['Category'].eq(category)
            & ember['Subcategory'].eq(subcategory)
            & ember['Variable'].eq(variable)
            & ember['Unit'].eq(unit)
        )
        sub = ember.loc[m, ['Alpha_3_code', 'Year', value_col]].copy()
        sub = sub.rename(columns={'Year': 'year', value_col: outvar})
        sub[outvar] = pd.to_numeric(sub[outvar], errors='coerce')
        frames.append(sub)
        add_meta(outvar, 'Ember yearly electricity data',
                 f'{category} - {subcategory} - {variable}.',
                 f'Exact long-format match on Category={category}, Subcategory={subcategory}, Variable={variable}, Unit={unit}; field={value_col}.',
                 unit if value_col == 'Value' else value_col)

    controls = reduce(lambda left, right: left.merge(right, on=['Alpha_3_code', 'year'], how='outer'), frames)
    controls = controls.merge(country_names, on='Alpha_3_code', how='left')

    controls = controls.sort_values(['Alpha_3_code', 'year']).copy()
    controls['demand_growth_3yr_pct'] = controls.groupby('Alpha_3_code')['demand_twh'].pct_change(3) * 100
    controls['demand_growth_5yr_pct'] = controls.groupby('Alpha_3_code')['demand_twh'].pct_change(5) * 100
    controls['net_imports_share_demand_pct'] = np.where(
        controls['demand_twh'].abs() > 0,
        controls['net_imports_twh'] / controls['demand_twh'] * 100,
        np.nan
    )
    total_capacity = controls['clean_capacity_gw'] + controls['fossil_capacity_gw']
    controls['coal_capacity_share_pct'] = np.where(total_capacity > 0, controls['coal_capacity_gw'] / total_capacity * 100, np.nan)
    controls['gas_capacity_share_pct'] = np.where(total_capacity > 0, controls['gas_capacity_gw'] / total_capacity * 100, np.nan)
    controls['renew_capacity_share_pct'] = np.where(total_capacity > 0, controls['renew_capacity_gw'] / total_capacity * 100, np.nan)
    controls['clean_capacity_share_pct'] = np.where(total_capacity > 0, controls['clean_capacity_gw'] / total_capacity * 100, np.nan)
    controls['fossil_capacity_share_pct'] = np.where(total_capacity > 0, controls['fossil_capacity_gw'] / total_capacity * 100, np.nan)

    derived = [
        ('demand_growth_3yr_pct', 'Three-year electricity demand growth.', '100 * (demand_twh / demand_twh lagged 3 years - 1).', '%'),
        ('demand_growth_5yr_pct', 'Five-year electricity demand growth.', '100 * (demand_twh / demand_twh lagged 5 years - 1).', '%'),
        ('net_imports_share_demand_pct', 'Net electricity imports as share of electricity demand.', '100 * net_imports_twh / demand_twh.', '%'),
        ('coal_capacity_share_pct', 'Coal capacity share in clean + fossil capacity.', '100 * coal_capacity_gw / (clean_capacity_gw + fossil_capacity_gw).', '%'),
        ('gas_capacity_share_pct', 'Gas capacity share in clean + fossil capacity.', '100 * gas_capacity_gw / (clean_capacity_gw + fossil_capacity_gw).', '%'),
        ('renew_capacity_share_pct', 'Renewables capacity share in clean + fossil capacity.', '100 * renew_capacity_gw / (clean_capacity_gw + fossil_capacity_gw).', '%'),
        ('clean_capacity_share_pct', 'Clean capacity share in clean + fossil capacity.', '100 * clean_capacity_gw / (clean_capacity_gw + fossil_capacity_gw).', '%'),
        ('fossil_capacity_share_pct', 'Fossil capacity share in clean + fossil capacity.', '100 * fossil_capacity_gw / (clean_capacity_gw + fossil_capacity_gw).', '%'),
    ]
    for var, desc, construction, unit in derived:
        add_meta(var, 'Derived from Ember yearly electricity data', desc, construction, unit)

    print(f'Ember controls: {controls.shape[0]:,} country-year rows, {controls.Alpha_3_code.nunique():,} countries')
    return controls

ember_controls = load_ember_controls()
ember_controls.head()

## 3. Net Zero Tracker target-strength controls

The Net Zero Tracker file is a current snapshot rather than a full historical panel. The notebook keeps current target attributes for documentation, but the intended Cox controls are the active-by-status-year variables: before a country's recorded status-update year they are set to 0, and from that year onward they take the target-strength/status values. This makes the policy target controls time-varying rather than a country-level constant absorbed by country strata.


In [ ]:
NETZERO_LIKE = {
    'Net zero', 'Carbon neutral(ity)', 'Climate neutral', 'GHG neutral(ity)',
    'Zero carbon', 'Zero emissions'
}
NEGATIVE_LIKE = {'Carbon negative', 'Net negative', 'Climate positive'}
REDUCTION_LIKE = {
    'Emissions reduction target', 'Reduction v. BAU', 'Emissions intensity target',
    'Absolute emissions target', 'Science-based target', '1.5°C target', 'Other'
}

STATUS_SCORE = {
    'Achieved (externally validated)': 4,
    'Achieved (self-declared)': 4,
    'In law': 4,
    'In policy document': 3,
    'Declaration / pledge': 2,
    'Proposed / in discussion': 1,
}


def target_type_score(target):
    if pd.isna(target) or target == 'No target':
        return 0
    if target in NEGATIVE_LIKE:
        return 3
    if target in NETZERO_LIKE:
        return 2
    if target in REDUCTION_LIKE:
        return 1
    return 1


def timing_factor(year):
    if pd.isna(year):
        return 0.0
    year = float(year)
    if year <= 2030:
        return 1.2
    if year <= 2040:
        return 1.1
    if year <= 2050:
        return 1.0
    if year <= 2060:
        return 0.8
    return 0.6


def load_netzero_controls():
    nz = pd.read_excel(NETZERO_INPUT, sheet_name='Current Snapshot', header=1)
    nz = nz[nz['Entity_type'].eq('Country')].copy()
    nz['Alpha_3_code'] = nz['Country'].map(clean_iso3)
    nz = nz.dropna(subset=['Alpha_3_code']).copy()

    out = pd.DataFrame({
        'Alpha_3_code': nz['Alpha_3_code'],
        'country_name_nz': nz['Name'],
        'nz_end_target_type': nz['End_target'].fillna('No target'),
        'nz_status': nz['Status_of_end_target'].fillna('No target'),
        'nz_end_year': pd.to_numeric(nz['End_target_year'], errors='coerce'),
        'nz_status_update_year': pd.to_numeric(nz['Date_of_last_status_update'], errors='coerce'),
        'nz_end_reduction_pct': pd.to_numeric(nz['End_target_percentage_reduction'], errors='coerce'),
        'nz_interim_year': pd.to_numeric(nz['Interim_target_year'], errors='coerce'),
        'nz_interim_reduction_pct': pd.to_numeric(nz['Interim_target_percentage_reduction'], errors='coerce'),
    })

    out['nz_has_target_current'] = (~out['nz_end_target_type'].isin(['No target', 'nan', 'None'])).astype(int)
    out['nz_netzero_like_current'] = out['nz_end_target_type'].isin(NETZERO_LIKE | NEGATIVE_LIKE).astype(int)
    out['nz_target_type_score'] = out['nz_end_target_type'].map(target_type_score).astype(float)
    out['nz_status_score'] = out['nz_status'].map(STATUS_SCORE).fillna(0).astype(float)
    out['nz_status_factor'] = (out['nz_status_score'] / 4).clip(0, 1)
    out['nz_timing_factor'] = out['nz_end_year'].map(timing_factor).astype(float)
    out['nz_reduction_pct_reported'] = out[['nz_end_reduction_pct', 'nz_interim_reduction_pct']].max(axis=1, skipna=True).fillna(0)
    out['nz_reduction_pct_imputed'] = out['nz_reduction_pct_reported']
    out.loc[out['nz_netzero_like_current'].eq(1), 'nz_reduction_pct_imputed'] = out.loc[
        out['nz_netzero_like_current'].eq(1), 'nz_reduction_pct_imputed'
    ].clip(lower=100)
    out.loc[out['nz_has_target_current'].eq(0), 'nz_reduction_pct_imputed'] = 0
    out['nz_strength_score_current'] = (
        (out['nz_target_type_score'] / 3).clip(0, 1) * 0.5
        + (out['nz_reduction_pct_imputed'] / 100).clip(0, 1) * 0.5
    ) * out['nz_status_factor'] * out['nz_timing_factor']
    out.loc[out['nz_has_target_current'].eq(0), 'nz_strength_score_current'] = 0

    out = out.drop_duplicates('Alpha_3_code')

    for var, desc, constr, unit, notes in [
        ('nz_has_target_current', 'Indicator for any current national end target in Net Zero Tracker.', '1 if End_target is not No target.', '', 'Current snapshot, repeated across years.'),
        ('nz_netzero_like_current', 'Indicator for net-zero, neutrality, zero-emissions, or net-negative style current target.', 'Target type belongs to net-zero-like or negative-like categories.', '', 'Current snapshot, repeated across years.'),
        ('nz_end_year', 'End target year.', 'Numeric End_target_year.', 'year', ''),
        ('nz_status_update_year', 'Year of last target status update.', 'Numeric Date_of_last_status_update.', 'year', 'Used only as approximate activation year for active variables.'),
        ('nz_end_reduction_pct', 'Reported end-target emissions reduction percentage.', 'Numeric End_target_percentage_reduction.', '%', 'Missing for many net-zero targets.'),
        ('nz_interim_reduction_pct', 'Reported interim-target emissions reduction percentage.', 'Numeric Interim_target_percentage_reduction.', '%', ''),
        ('nz_reduction_pct_reported', 'Maximum reported end or interim reduction percentage.', 'max(end reduction pct, interim reduction pct), missing set to 0.', '%', 'Does not impute net zero as 100.'),
        ('nz_reduction_pct_imputed', 'Reduction percentage with net-zero-like targets imputed as at least 100.', 'max(reported reduction, 100) for net-zero-like targets; zero for no target.', '%', 'Use as a simple target ambition proxy.'),
        ('nz_status_score', 'Legal/policy status score for current end target.', 'Achieved/In law=4, policy document=3, pledge=2, proposed=1, no target=0.', 'index', ''),
        ('nz_target_type_score', 'Target type score.', 'No target=0, reduction/intensity/BAU/other=1, net-zero/neutrality/zero=2, negative/positive=3.', 'index', ''),
        ('nz_strength_score_current', 'Composite current target strength score.', 'Average of normalized target-type and imputed reduction ambition, multiplied by status and timing factors.', 'index', 'Ad hoc but transparent; components are also exported separately.'),
    ]:
        add_meta(var, 'Net Zero Tracker current snapshot', desc, constr, unit, notes)

    print(f'Net-zero controls: {out.shape[0]:,} countries')
    return out

netzero_controls = load_netzero_controls()
netzero_controls.head()

## 4. Climate Policy Database controls

The reviewer asked for policy controls where available. This notebook counts new policies by country-year using `decision_date`, and also creates cumulative counts through each year. The cumulative variables better approximate the stock of policy environment facing coal plants and data centers.

In [ ]:
def policy_count_frame(df, condition, outvar):
    sub = df.loc[condition, ['Alpha_3_code', 'decision_year']].copy()
    if sub.empty:
        return pd.DataFrame(columns=['Alpha_3_code', 'year', outvar])
    out = sub.groupby(['Alpha_3_code', 'decision_year']).size().reset_index(name=outvar)
    return out.rename(columns={'decision_year': 'year'})


def make_cumulative(panel, annual, var):
    tmp = panel[['Alpha_3_code', 'year']].merge(annual[['Alpha_3_code', 'year', var]], on=['Alpha_3_code', 'year'], how='left')
    tmp[var] = tmp[var].fillna(0)
    tmp = tmp.sort_values(['Alpha_3_code', 'year'])
    cumvar = safe_stata_name(var.replace('_new', '_cum'))
    tmp[cumvar] = tmp.groupby('Alpha_3_code', sort=False)[var].cumsum()
    return tmp[['Alpha_3_code', 'year', cumvar]]


def load_policy_controls(panel):
    alpha_codes = panel['Alpha_3_code'].dropna().astype(str).unique()
    panel_years = panel[['Alpha_3_code', 'year']].copy()

    pol = pd.read_csv(POLICY_INPUT)
    pol['Alpha_3_code'] = pol['country_iso'].map(clean_iso3)
    pol['decision_year'] = pd.to_numeric(pol['decision_date'], errors='coerce')
    pol = pol.dropna(subset=['Alpha_3_code', 'decision_year']).copy()
    pol['decision_year'] = pol['decision_year'].astype(int)
    high_impact = pol['high_impact'].astype(str).str.lower().str.contains('high', na=False)
    mitigation = pol['policy_objective'].astype(str).str.lower().str.contains('mitigation', na=False)
    electricity = pol['sector'].astype(str).str.lower().str.contains('electricity and heat', na=False)
    in_force = pol['policy_status'].astype(str).str.lower().eq('in force')

    annual_frames = [
        policy_count_frame(pol, pd.Series(True, index=pol.index), 'policy_count_new'),
        policy_count_frame(pol, mitigation, 'mitigation_policy_count_new'),
        policy_count_frame(pol, electricity, 'electricity_policy_count_new'),
        policy_count_frame(pol, high_impact, 'high_impact_policy_count_new'),
        policy_count_frame(pol, in_force, 'inforce_policy_count_new'),
    ]

    annual = reduce(lambda l, r: l.merge(r, on=['Alpha_3_code', 'year'], how='outer'), annual_frames)
    controls = panel_years.merge(annual, on=['Alpha_3_code', 'year'], how='left')

    count_cols = [c for c in controls.columns if c.endswith('_count_new')]
    controls[count_cols] = controls[count_cols].fillna(0)

    cum_frames = []
    annual_all_years = reduce(lambda l, r: l.merge(r, on=['Alpha_3_code', 'year'], how='outer'), annual_frames)
    full_panel = make_panel(alpha_codes, min(int(pol['decision_year'].min()), int(panel_years['year'].min())), int(panel_years['year'].max()))
    for var in [c for c in annual_all_years.columns if c.endswith('_count_new')]:
        cum_frames.append(make_cumulative(full_panel, annual_all_years[['Alpha_3_code', 'year', var]], var))

    cumulative = reduce(lambda l, r: l.merge(r, on=['Alpha_3_code', 'year'], how='outer'), cum_frames)
    controls = controls.merge(cumulative, on=['Alpha_3_code', 'year'], how='left')

    cum_cols = [c for c in controls.columns if c.endswith('_count_cum')]
    controls[cum_cols] = controls[cum_cols].fillna(0)

    meta = [
        ('policy_count_new', 'Number of climate policies decided in the country-year.', 'Count of policies by country_iso and decision_date year.'),
        ('mitigation_policy_count_new', 'Number of mitigation-related policies decided in the country-year.', 'Count where policy_objective contains mitigation.'),
        ('electricity_policy_count_new', 'Number of electricity-and-heat policies decided in the country-year.', 'Count where sector contains Electricity and heat.'),
        ('high_impact_policy_count_new', 'Number of high-impact policies decided in the country-year.', 'Count where high_impact contains high.'),
        ('inforce_policy_count_new', 'Number of policies currently marked In force that were decided in the country-year.', 'Count where policy_status is In force.'),
        ('policy_count_cum', 'Cumulative number of climate policies decided through the country-year.', 'Cumulative sum of policy_count_new including policies before 2000.'),
        ('mitigation_policy_count_cum', 'Cumulative number of mitigation policies through the country-year.', 'Cumulative sum of mitigation_policy_count_new including policies before 2000.'),
        ('electricity_policy_count_cum', 'Cumulative number of electricity-and-heat policies through the country-year.', 'Cumulative sum including policies before 2000.'),
        ('high_impact_policy_count_cum', 'Cumulative number of high-impact policies through the country-year.', 'Cumulative sum including policies before 2000.'),
        ('inforce_policy_count_cum', 'Cumulative number of currently in-force policies decided through the country-year.', 'Cumulative sum including policies before 2000.'),
    ]
    for var, desc, constr in meta:
        add_meta(var, 'Climate Policy Database', desc, constr, 'count/index')

    print(f'Policy controls: {controls.shape[0]:,} country-year rows, {controls.Alpha_3_code.nunique():,} countries')
    return controls

# policy_controls is built after alpha-code universe is known.
print('Policy helper ready.')


## 5. Merge and export country-year controls

The final panel uses country-specific year coverage. For each country, the notebook finds the earliest and latest year observed in any control-data source, then keeps all calendar years between those two endpoints. The year range is therefore not a fixed global window; it is determined by each country's own observed control-data coverage.


In [ ]:
def build_controls():
    year_bounds = infer_country_year_bounds(
        price_controls,
        td_loss_controls,
        ember_controls,
        netzero_controls[['Alpha_3_code', 'nz_status_update_year']].rename(columns={'nz_status_update_year': 'year'}),
        pd.read_csv(POLICY_INPUT, usecols=['country_iso', 'decision_date']).assign(
            Alpha_3_code=lambda x: x['country_iso'].map(clean_iso3),
            year=lambda x: pd.to_numeric(x['decision_date'], errors='coerce')
        )[['Alpha_3_code', 'year']]
    )

    panel = make_country_specific_panel(year_bounds)
    print(
        f'Base country-year panel: {panel.shape[0]:,} rows, '
        f'{panel.Alpha_3_code.nunique():,} countries; '
        f'country-specific years {panel.year.min():.0f}-{panel.year.max():.0f}'
    )

    policy_controls = load_policy_controls(panel)

    df = panel.copy()
    df = df.merge(price_controls, on=['Alpha_3_code', 'year'], how='left')
    df = df.merge(td_loss_controls, on=['Alpha_3_code', 'year'], how='left')
    df = df.merge(ember_controls, on=['Alpha_3_code', 'year'], how='left')
    df = df.merge(netzero_controls, on='Alpha_3_code', how='left')
    df = df.merge(policy_controls, on=['Alpha_3_code', 'year'], how='left')

    # Keep electricity prices exactly as reported at the national country-year level.
    # Missing values are intentionally left missing for later user-defined filling choices.
    df = df.sort_values(['Alpha_3_code', 'year']).copy()

    # Net-zero active variables using status update year as approximate activation year.
    has_status_year = df['nz_status_update_year'].notna()
    active = has_status_year & (df['year'] >= df['nz_status_update_year'])
    for col in ['nz_has_target_current', 'nz_netzero_like_current', 'nz_strength_score_current', 'nz_reduction_pct_imputed', 'nz_status_score']:
        if col in df.columns:
            active_col = col.replace('_current', '_active') if col.endswith('_current') else col + '_active'
            active_col = safe_stata_name(active_col)
            df[active_col] = np.where(active, df[col].fillna(0), 0)
            add_meta(active_col, 'Derived from Net Zero Tracker current snapshot',
                     f'Approximate active-by-year version of {col}.',
                     f'{col} if year >= Date_of_last_status_update; otherwise 0.',
                     notes='This is an approximation because the source is a current snapshot, not a full historical target database.')

    # Fill zero-valued policy variables where the country-year has no policy record.
    for col in [c for c in df.columns if c.endswith('_count_new') or c.endswith('_count_cum')]:
        df[col] = df[col].fillna(0)

    # Missing net-zero current indicators are set to zero for countries absent from the source.
    zero_if_missing = [
        'nz_has_target_current', 'nz_netzero_like_current', 'nz_target_type_score',
        'nz_status_score', 'nz_status_factor', 'nz_timing_factor',
        'nz_reduction_pct_reported', 'nz_reduction_pct_imputed',
        'nz_strength_score_current'
    ]
    for col in zero_if_missing:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # Add one-year lags for the core variables most likely to be used in Cox robustness models.
    lag_vars = [
        'elec_price_db_uscent_kwh_raw', 'td_loss_pct_output',
        'demand_twh', 'demand_yoy_pct', 'demand_growth_3yr_pct',
        'renew_gen_share_pct', 'wind_solar_gen_share_pct', 'clean_gen_share_pct',
        'coal_gen_share_pct', 'gas_gen_share_pct', 'co2_intensity_g_kwh',
        'coal_capacity_share_pct', 'gas_capacity_share_pct', 'renew_capacity_share_pct',
        'policy_count_new', 'policy_count_cum', 'mitigation_policy_count_cum',
        'electricity_policy_count_cum', 'high_impact_policy_count_cum',
        'nz_strength_score_current', 'nz_strength_score_active', 'nz_has_target_active'
    ]
    df = add_lags(df, lag_vars, lag=1)

    # Stata export hygiene.
    for col in df.columns:
        safe_stata_name(col)

    df = df.sort_values(['Alpha_3_code', 'year']).reset_index(drop=True)
    dict_df = pd.DataFrame(metadata_rows).drop_duplicates(subset=['variable'], keep='first')
    coverage_df = build_coverage(df)

    base_labels = {
        'Alpha_3_code': 'ISO 3166-1 alpha-3 country code',
        'year': 'Calendar year',
        'country_name_price': 'Country name in Doing Business electricity price file',
        'country_name_nz': 'Country name in Net Zero Tracker',
        'nz_end_target_type': 'Net Zero Tracker current end-target type',
        'nz_status': 'Net Zero Tracker current target status',
    }
    metadata_labels = dict(zip(dict_df['variable'], dict_df['description'])) if not dict_df.empty else {}

    def stata_label(text, max_len=80):
        text = re.sub(r'\s+', ' ', str(text)).strip()
        text = text.replace('CO₂', 'CO2').replace('–', '-').replace('—', '-')
        if len(text) <= max_len:
            return text
        return text[:max_len - 1].rstrip() + '.'

    variable_labels = {}
    for col in df.columns:
        label = base_labels.get(col) or metadata_labels.get(col) or col.replace('_', ' ')
        variable_labels[col] = stata_label(label)

    df.to_csv(OUTPUT_CSV, index=False)
    df.to_stata(OUTPUT_DTA, write_index=False, version=118, variable_labels=variable_labels)
    df.to_stata(OUTPUT_USE_DTA, write_index=False, version=118, variable_labels=variable_labels)
    dict_df.to_csv(DICT_OUTPUT, index=False)
    coverage_df.to_csv(COVERAGE_OUTPUT, index=False)

    print('\nSaved controls:')
    print(f'  {OUTPUT_CSV}')
    print(f'  {OUTPUT_DTA}')
    print(f'  {OUTPUT_USE_DTA}')
    print(f'  {DICT_OUTPUT}')
    print(f'  {COVERAGE_OUTPUT}')
    print(f'Final shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')

    return df, dict_df, coverage_df

controls_df, dictionary_df, coverage_df = build_controls()
controls_df.head()


In [ ]:
print('Selected control coverage:')
selected = [
    'elec_price_db_uscent_kwh_raw', 'td_loss_pct_output',
    'demand_twh', 'demand_yoy_pct', 'renew_gen_share_pct', 'coal_gen_share_pct',
    'co2_intensity_g_kwh', 'policy_count_cum', 'electricity_policy_count_cum',
    'nz_strength_score_current', 'nz_strength_score_active'
]
print(coverage_df[coverage_df['variable'].isin(selected)].sort_values('variable').to_string(index=False))

print()
print('Example rows for USA, CHN, IND, DEU, RUS in 2020-2024:')
example_cols = [
    'Alpha_3_code', 'year', 'elec_price_db_uscent_kwh_raw', 'td_loss_pct_output', 'demand_yoy_pct',
    'renew_gen_share_pct', 'coal_gen_share_pct', 'co2_intensity_g_kwh',
    'policy_count_cum', 'electricity_policy_count_cum',
    'nz_strength_score_current', 'nz_strength_score_active'
]
print(controls_df[(controls_df['Alpha_3_code'].isin(['USA','CHN','IND','DEU','RUS'])) & (controls_df['year'].between(2020, 2024))][example_cols].to_string(index=False))
